### Importation des librairies

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from tqdm import tqdm
import warnings
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from joblib import Parallel, delayed



### Importation des fichiers

In [2]:
x_train = pd.read_csv('data/x_train.csv', index_col=0)
x_test = pd.read_csv('data/x_test.csv', index_col=0)
y_train = pd.read_csv('data/y_train.csv', index_col=0)
sample_submission = pd.read_csv('data/new_output_sample.csv', index_col=0)

print("Dimensions x_train :", x_train.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions x_test :", x_test.shape)
print("Dimensions sample_submission :", sample_submission.shape)

Dimensions x_train : (1057, 21000)
Dimensions y_train : (1057, 1000)
Dimensions x_test : (1057, 38140)
Dimensions sample_submission : (1057, 1000)


### Fonctions utiles

In [3]:
# Génération du fichier de soumission
def generer_soumission(soumission, nom_sortie):
    is_valid = (soumission.shape == sample_submission.shape) # Vérification du format avec le format cible
    if is_valid:
        soumission.to_csv(f'{nom_sortie}.csv') # Génération du fichier de soumission selon le nom entré en paramètre
        print(f"Fichier '{nom_sortie}.csv' généré avec succès !") 
    else:
        print("Attention, les dimensions ne correspondent pas au fichier sample.")

In [4]:
def benchmark(column):
    col = column.copy()
    col = col.interpolate(method='linear', limit_direction='both')
    return col

In [5]:
def echantilloner (nb_echantillon):
    # Si none alors pas d'échantillonage
    holed_cols = [col for col in x_test.columns if 'holed' in col]
    complete_cols = [col for col in x_test.columns if 'holed' not in col]
    x_test_filled = x_test[holed_cols].copy()
    if nb_echantillon != None :
        X_features = x_test[complete_cols].sample(n=nb_echantillon, axis=1, random_state=67) # Permet de faire un échantillon parmis toutes les données pour accélérer le calcul (le calcul de base prend environ 1h sans échantillon)
    else :
        X_features = x_test[complete_cols]
    return holed_cols, x_test_filled, X_features

### Soumission de base (interpolation linéaire)

Fonction d'interpolation linéaire (remplissage des trous par une ligne droite, comme lors des précédents TP, pour éviter une erreur nan)

In [6]:
def interpolation_lineaire(column):
    return column.interpolate(method='linear', limit_direction='both') # Evite les erreurs nan comme lors des derniers TP

In [7]:
holed_cols_test = [col for col in x_test.columns if 'holed' in col]
y_pred_test = x_test[holed_cols_test].apply(interpolation_lineaire, axis=0)
submission = y_pred_test.loc[sample_submission.index, sample_submission.columns]

generer_soumission(submission, "interpolation_lineaire")

Fichier 'interpolation_lineaire.csv' généré avec succès !


### Régression linéaire
Le score donné par cette regression linéaire est de 93 (mais nécessite les données complètes et ça tourne pendant un peu plus d'une heure)

Avec un échantillon de 6000 et un head à 30 on obtient un score de 98 avec un temps d'exécution de 15 min environ

In [ ]:
def regression_lineaire():

    holed_cols, x_test_filled, X_features_reduced = echantilloner(None)
    print("Lancement de la régression linéaire sur l'échantillon...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        # Enlève les messages d'erreur inutiles
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        model = LinearRegression()
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "regression_lineaire_full")
    return submission

regression_lineaire()



### Arbres

Fonctionne moins bien qu'une régression linéaire (106 contre 103 pour 2000 échantillons)

In [ ]:
def regression_arbre_decision():
    holed_cols, x_test_filled, X_features_reduced = echantilloner(2000)

    print("Lancement de l'Arbre de Décision (Sécurité anti-valeurs aberrantes)...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(15).index
        
        # Arbre max_depth = 6
        model = DecisionTreeRegressor(max_depth=6, random_state=67)
        
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])
        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    
    generer_soumission(submission, "arbre_decision_depth6")
    
    return submission

ma_soumission_arbre = regression_arbre_decision()

### Réseau de neurone
Version basée sur le dernier TP avec Tensorflow, elle n'est pas bien optimisée pour le texte et obtient un score de 94 pour 1h30 de calcul

In [ ]:
def regression_reseau_neurones_opti():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(6000)

    print("Lancement du Réseau de Neurones...")
    
    tf.get_logger().setLevel('ERROR')

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train_nn = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        scaler = StandardScaler() # Normalisation
        X_train_nn = scaler.fit_transform(X_train_raw)
        X_missing_nn = scaler.transform(X_missing_raw)
        # -----------------------------------------
        
        model = Sequential([Dense(64, activation='relu', input_shape=(len(top_vars),)), Dense(32, activation='relu'), Dense(1)])
        
        model.compile(optimizer='adam', loss='mae')
        
        early_stop = EarlyStopping(monitor='loss', patience=10, verbose=0) # Augmentation de la patiente par rapport à la dernière itération
        
        model.fit(X_train_nn, y_train_nn, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
        
        predictions = model.predict(X_missing_nn, verbose=0)
        
        x_test_filled.loc[mask_missing, target_col] = predictions.flatten()

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    
    generer_soumission(submission, "reseau_neurones_keras_tensorflow")
    
    return submission

ma_soumission_finale = regression_reseau_neurones_opti()

Version utilisant Sklearn, plus adaptée pour du texte et le contexte du hackaton. Score de 83 en 25 min de calcul

In [ ]:

def regression_gradient_boosting_features():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(None) # A modifier pour affinement

    print("Lancement du Gradient Boosting (Arbres) + Feature Engineering...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        
        # 50 meilleures courbes (paramètre à modifier pour afiner)
        top_vars = corrs.sort_values(ascending=False).head(100).index
        
        # Extraction des données brutes
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        # Création de nouvelles colones pour optimiser l'algo
        def ajouter_features_statistiques(X):
            moyenne = np.mean(X, axis=1, keepdims=True)
            ecart_type = np.std(X, axis=1, keepdims=True)
            maximum = np.max(X, axis=1, keepdims=True)
            minimum = np.min(X, axis=1, keepdims=True)
            # Collage des nouvelles colonnes avec les 50 courbes
            return np.hstack((X, moyenne, ecart_type, maximum, minimum))
            
        X_train_enriched = ajouter_features_statistiques(X_train_raw)
        X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
        
        model = HistGradientBoostingRegressor(
            loss='absolute_error', # Minimise l'erreur
            max_iter=1000,         
            learning_rate=0.03,    
            max_depth=6, # Arbres pas trop profonds
            early_stopping=True, # Limite le temps d'exécution en arrêtant le programme si l'apprentissage ne s'améliore plus
            validation_fraction=0.1, # Garde 10% des données pour tester l'arrêt
            n_iter_no_change=20,
            random_state=67
        )
        
        model.fit(X_train_enriched, y_train)
        predictions = model.predict(X_missing_enriched)
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "reseau_neurones_sklearn_full")
    
    return submission

ma_soumission_elite = regression_gradient_boosting_features()

In [ ]:
# Séparation du code pour exécution parallèle
def predire_une_colonne_opti(target_col, x_test_global, X_features_global):
    target_series = x_test_global[target_col]
    mask_known = target_series.notna()
    mask_missing = target_series.isna()
    
    if mask_known.sum() < 2 or not mask_missing.any():
        return target_col, None, None 

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        corrs = X_features_global[mask_known].corrwith(target_series[mask_known]).abs()
    
    corrs = corrs.fillna(0)
    
    # Garde les 100 meilleures courbes
    top_vars = corrs.sort_values(ascending=False).head(100).index
    
    X_train_raw = X_features_global.loc[mask_known, top_vars].values
    y_train = target_series[mask_known].values
    X_missing_raw = X_features_global.loc[mask_missing, top_vars].values
    
    # --- OPTIMISATION 1 : LES QUANTILES ---
    def ajouter_features_statistiques(X):
        moyenne = np.mean(X, axis=1, keepdims=True)
        mediane = np.median(X, axis=1, keepdims=True)
        ecart_type = np.std(X, axis=1, keepdims=True)
        maximum = np.max(X, axis=1, keepdims=True)
        minimum = np.min(X, axis=1, keepdims=True)
        # Nouveaux indicateurs robustes au bruit
        q25 = np.percentile(X, 25, axis=1, keepdims=True)
        q75 = np.percentile(X, 75, axis=1, keepdims=True)
        
        return np.hstack((X, moyenne, mediane, ecart_type, maximum, minimum, q25, q75))
        
    X_train_enriched = ajouter_features_statistiques(X_train_raw)
    X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
    
    # --- OPTIMISATION 2 : RÉGULARISATION ET PROFONDEUR ---
    model = HistGradientBoostingRegressor(
        loss='absolute_error', 
        max_iter=3000,        # Un peu plus de temps pour compenser la profondeur réduite
        learning_rate=0.01,   # Apprentissage encore plus doux
        max_depth=6,          # Réduit pour éviter l'apprentissage par coeur (overfitting)
        l2_regularization=0.5,# FORCE l'algo à ne pas trop s'appuyer sur un seul voisin
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=40,  # Plus de patience
        random_state=67
    )
    
    model.fit(X_train_enriched, y_train)
    predictions = model.predict(X_missing_enriched)
    
    return target_col, predictions, mask_missing

# Fonction principale
def regression_gradient_boosting_parallele_ultime():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(None)

    print("Lancement du Gradient Boosting en parallèle")

    # Traitement en parallèle (-1 pour utiliser le processeur complètement)
    resultats = Parallel(n_jobs=-1)(
        delayed(predire_une_colonne_opti)(col, x_test, X_features_reduced) for col in tqdm(holed_cols)
    )
    
    # Assemblage
    print("Assemblage...")
    for target_col, predictions, mask_missing in resultats:
        if predictions is not None:
            x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "Gradient_boosting_Parallele_100_Courbes")
    
    return submission

ma_soumission_absolue = regression_gradient_boosting_parallele_ultime()

In [ ]:
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor

# Séparation du code pour exécution parallèle
def predire_une_colonne_opti(target_col, x_test_global, X_features_global):
    target_series = x_test_global[target_col]
    mask_known = target_series.notna()
    mask_missing = target_series.isna()
    
    # S'il y a moins de 5 points pour apprendre, l'algorithme va dire n'importe quoi.
    if mask_known.sum() < 5 or not mask_missing.any():
        return target_col, None, None 

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        corrs = X_features_global[mask_known].corrwith(target_series[mask_known]).abs()
    
    corrs = corrs.fillna(0)
    
    # Garde les 100 meilleures courbes
    top_vars = corrs.sort_values(ascending=False).head(100).index
    
    X_train_raw = X_features_global.loc[mask_known, top_vars].values
    y_train = target_series[mask_known].values
    X_missing_raw = X_features_global.loc[mask_missing, top_vars].values
    
    # --- OPTIMISATION 1 : STATISTIQUES BLINDÉES CONTRE LES NaNs ---
    def ajouter_features_statistiques(X):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Utilisation des versions "nan" pour ignorer les trous dans les courbes corrélées
            moyenne = np.nanmean(X, axis=1, keepdims=True)
            mediane = np.nanmedian(X, axis=1, keepdims=True)
            ecart_type = np.nanstd(X, axis=1, keepdims=True)
            maximum = np.nanmax(X, axis=1, keepdims=True)
            minimum = np.nanmin(X, axis=1, keepdims=True)
            q25 = np.nanpercentile(X, 25, axis=1, keepdims=True)
            q75 = np.nanpercentile(X, 75, axis=1, keepdims=True)
        
        features = np.hstack((X, moyenne, mediane, ecart_type, maximum, minimum, q25, q75))
        # Si une ligne était 100% vide, on remplace les NaNs résiduels par 0
        return np.nan_to_num(features, nan=0.0)
        
    X_train_enriched = ajouter_features_statistiques(X_train_raw)
    X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
    
    # --- OPTIMISATION 2 : ANTI-OVERFITTING POUR PETIT DATASET ---
    model = HistGradientBoostingRegressor(
        loss='absolute_error', # Note : Si le hackathon Engie évalue avec la métrique RMSE, remplace par 'squared_error' !
        max_iter=1000,         # Réduit : 3000 est trop pour un petit dataset, le modèle va apprendre le bruit par coeur.
        learning_rate=0.025,    # Légèrement plus rapide pour compenser
        max_depth= 7,           # Réduit de 6 à 5 : force des règles plus simples et plus générales
        l2_regularization=0.1, # AUGMENTÉ : Empêche un seul voisin de monopoliser la décision
        random_state=67
    )
    
    model.fit(X_train_enriched, y_train)
    predictions = model.predict(X_missing_enriched)
    
    return target_col, predictions, mask_missing

# Fonction principale
def regression_gradient_boosting_parallele_ultime():
    # Assure-toi que cette fonction renvoie bien des objets avec .notna() et pas des array numpy nus
    holed_cols, x_test_filled, X_features_reduced = echantilloner(6000)

    print("Lancement du Gradient Boosting en parallèle")

    print(f"Lancement du Gradient Boosting sur les {len(holed_cols)} colonnes...")
    
    resultats = []
    
    # 1. On crée une barre de progression vide de la taille du nombre de colonnes
    with tqdm(total=len(holed_cols), desc="Prédictions") as pbar:
        
        # 2. Ajout de return_as="generator" : Parallel renvoie les résultats un par un dès qu'ils sont prêts
        generateur = Parallel(n_jobs=4, return_as="generator")(
            delayed(predire_une_colonne_opti)(col, x_test_filled, X_features_reduced) for col in holed_cols
        )
        
        # 3. On récupère les résultats et on fait avancer la barre d'un cran
        for res in generateur:
            resultats.append(res)
            pbar.update(1)  # Met à jour le temps estimé !
    
    # Assemblage
    print("Assemblage...")
    for target_col, predictions, mask_missing in resultats:
        if predictions is not None:
            x_test_filled.loc[mask_missing, target_col] = predictions

    # Assure-toi que sample_submission est bien défini globalement dans ton code
    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "Gradient_boosting_Parallele_100_Courbes_Opti")
    
    return submission

ma_soumission_absolue = regression_gradient_boosting_parallele_ultime()

Lancement du Gradient Boosting en parallèle
Lancement du Gradient Boosting sur les 1000 colonnes...


Prédictions:   1%|          | 10/1000 [00:25<41:29,  2.51s/it] 


KeyboardInterrupt: 